# GLiNER2: 개념과 동작 구조

본 노트북은 GLiNER2의 추출 방식과 구성 요소를 정의한다. 분석 대상은 `gliner2` 2.1.0 패키지와 GLiNER2.5 체크포인트(`fastino/gliner2.5-base-v1`, `fastino/gliner2.5-multi-v1`)이다. 설명은 패키지에 포함된 [공식 README](https://github.com/fastino-ai/GLiNER2)와 설치된 소스를 근거로 하며, 한국어 입력에 관한 관찰은 이 프로젝트의 실험 결과를 인용한다.

주요 논점은 추출 방식, 아키텍처의 계보, 스키마, 단어 분할, 디코딩과 threshold, 학습이다. 사용법은 [기본 사용법](01_basic_usage.ipynb), 한국어 데이터 학습은 [한국어 미세조정](02_korean_fine_tuning.ipynb)에서 다룬다.

## 개체명 인식의 세 가지 접근

개체명 인식(NER)은 문장에서 사람·조직·장소 같은 이름을 찾아 범위(span)와 타입을 붙이는 작업이다. 현재 쓰이는 방식은 크게 세 가지로 나뉜다.

| 방식 | 동작 | 타입 집합 | 대표 예 |
|---|---|---|---|
| 시퀀스 레이블링 | 토큰마다 `B-PER`, `I-PER`, `O` 같은 태그를 분류한다. | 학습 시 고정 | BERT-CRF, spaCy NER, KLUE-NER 모델 |
| 생성형 LLM | 지시문으로 타입을 설명하고 결과를 텍스트(JSON)로 생성하게 한다. | 실행 시 자유롭게 지정 | GPT, Claude 등 |
| 스팬–레이블 매칭 | 문장의 후보 범위와 타입 이름을 같은 공간에 표현하고, 둘의 일치 점수로 엔티티를 고른다. | 실행 시 자유롭게 지정 | GLiNER 계열 |

GLiNER 계열은 세 번째 방식이다. 시퀀스 레이블링처럼 작은 인코더 모델로 빠르게 동작하면서, LLM처럼 실행 시점에 원하는 타입을 이름(과 설명)으로 지정할 수 있다. 대신 결과를 생성하지 않고 원문의 범위만 고르므로, 문장에 없는 표현을 만들어 내지 않는 반면 문장 밖의 지식이나 함축된 의미를 추론하지도 못한다.

## 아키텍처의 계보

### GLiNER

[GLiNER](https://arxiv.org/abs/2311.08526)(NAACL 2024)는 타입 이름들과 문장을 하나의 입력으로 이어 붙여 양방향 Transformer 인코더(BERT 계열)에 넣는다. 인코더 출력에서 타입마다 하나의 임베딩과 후보 범위마다 하나의 임베딩을 만들고, 두 임베딩의 일치 점수가 threshold를 넘는 범위를 해당 타입의 엔티티로 고른다.

```text
[타입1] [타입2] ... [SEP] 문 장 의 단 어 들
   │       │                 │
   ▼       ▼                 ▼
 타입 임베딩              범위(span) 임베딩
        └──── 일치 점수 ────┘  → threshold 이상이면 엔티티
```

모든 타입과 모든 후보 범위를 한 번의 인코딩으로 동시에 평가하므로, 토큰을 하나씩 생성하는 LLM보다 훨씬 빠르다.

#### 일치 점수는 코사인 유사도가 아니다

일치 점수는 `sigmoid(범위 임베딩 · 타입 임베딩)`이다. 내적이라는 점에서는 유사도 계산과 같지만, 범용 임베딩의 코사인 유사도와는 두 가지가 다르다.

- **임베딩을 판정 목적으로 학습한다.** 범용 임베딩은 "관련 있음"을 재므로 `무신사`는 `organization`보다 `패션`에 가깝다. GLiNER는 학습 문장에서 정답 (범위, 타입) 쌍의 점수를 1로, 나머지 모든 쌍을 0으로 밀어서 "이 범위가 이 타입의 한 예인가"를 판정하도록 인코더와 투영층을 학습한다.
- **점수를 예/아니오 확률로 읽는다.** 코사인 값은 순위만 알려 주고 기준선이 없다. GLiNER는 타입마다 독립적으로 점수를 내고 0.5 같은 공통 기준으로 자르므로, 한 범위가 어떤 타입에도 해당하지 않거나 여러 타입에 해당할 수 있다.

또한 타입 이름과 문장을 함께 인코딩하므로, 같은 단어도 문맥에 따라 다른 범위 임베딩을 갖고, 타입 목록이 바뀌면 기존 타입의 점수도 달라진다([기본 사용법](01_basic_usage.ipynb)의 레이블 집합 예시).

> **범용 임베딩 + 코사인은 "가장 비슷한 것 찾기"이고, GLiNER는 "레이블마다 예/아니오로 답하도록 학습된 유사도"이다.**

다만 학습된 점수라도 실제 정답률과 맞도록 보정된 것은 아니다. confidence 0.9가 "90% 확률로 맞다"는 뜻은 아니며, 틀린 답이 맞은 답보다 높은 점수를 받기도 한다(아래 Threshold 참고).

### GLiNER2

[GLiNER2]({README})는 같은 원리를 **스키마 조건부 다중 작업**으로 확장했다. 하나의 스키마에 엔티티, 문장 분류, 구조화 레코드(JSON), 관계를 함께 선언하고 한 번의 순전파로 모두 추출한다. 패키지는 두 아키텍처를 같은 API 뒤에 둔다.

| 아키텍처 | 클래스 | 후보 범위 표현 | 체크포인트 |
|---|---|---|---|
| `span` | `GLiNER2` / `SpanExtractor` | 고정 길이까지의 모든 범위를 격자로 평가 | `fastino/gliner2-*-v1`, PII·Guardrails 모델 |
| `boundary` | `BoundaryExtractor` (**GLiNER2.5**) | 시작·끝 위치를 희소하게 짝지어 평가. 인코딩 창 안이면 길이 제한 없음 | `fastino/gliner2.5-*-v1` |

`AutoExtractor.from_pretrained()`는 체크포인트에 저장된 `architecture` 값을 보고 알맞은 클래스로 불러온다. `GLiNER2.from_pretrained()`는 span 전용이라 GLiNER2.5 체크포인트를 불러오지 못한다.

### GLiNER2.5 체크포인트

| 체크포인트 | 파라미터 | 인코더 | 언어 |
|---|---|---|---|
| `fastino/gliner2.5-small-v1` | 74M | DeBERTa-v3-xsmall | 영어 |
| `fastino/gliner2.5-base-v1` | 194M | DeBERTa-v3-base | 영어 |
| `fastino/gliner2.5-multi-v1` | 287M | mDeBERTa-v3-base | 다국어 |

공식 문서는 영어에 `base`, 다국어에 `multi`를 권장한다. 한국어 문장 `JeekLee는 무신사에서 일하고, 주말에는 쿠팡에서 장을 본다.`에 person, organization, location 레이블을 주면 두 체크포인트의 결과가 갈린다.

| 체크포인트 | person | organization | location |
|---|---|---|---|
| `base` | JeekLee | 쿠팡 (0.53) | **쿠팡** (0.71) |
| `multi` | JeekLee | 무신사 (0.97), 쿠팡 (0.92) | – |

`base`는 `무신사`를 찾지 못하고 `쿠팡`을 장소로 더 높게 본다. 영어 위주로 학습한 체크포인트는 한국어 고유명사의 타입을 판단할 근거가 부족하기 때문이다. 그래서 한국어에서는 `multi`를 사용한다. 더 많은 문장으로 비교한 결과는 [언어와 체크포인트 비교](../experiments/01_language_and_checkpoint.ipynb)에 있다.

## 스키마

GLiNER2에서 "무엇을 추출할지"는 모델이 아니라 **스키마**가 정한다. 스키마는 실행할 때마다 바꿀 수 있으며, 같은 모델로 서로 다른 타입 집합을 추출할 수 있다.

| 작업 | 간편 메서드 | 스키마 빌더 | 결과 |
|---|---|---|---|
| 엔티티 | `extract_entities(text, labels)` | `.entities(...)` | 타입별 범위 목록 |
| 분류 | `classify_text(text, tasks)` | `.classification(task, labels)` | 작업별 레이블 |
| 구조화 레코드 | `extract_json(text, structures)` | `.structure(name).field(...)` | 필드를 가진 레코드 목록 |
| 관계 | `extract_relations(text, relations)` | `.relations(...)` | 관계별 (머리, 꼬리) 쌍 |
| 결합 | `extract(text, schema)` | 위 빌더를 이어서 호출 | 모든 작업의 결과를 한 번에 |

레이블은 이름만 줄 수도 있고 `{"이름": "설명"}`으로 설명을 붙일 수도 있다. 설명은 레이블 임베딩의 입력이 되므로, 이름만으로 모호한 타입(예: `organization`이 회사만인지 정부 기관도 포함하는지)의 경계를 정하는 데 쓰인다.

레이블 집합 자체도 결과에 영향을 준다. 각 범위는 주어진 레이블 중에서 타입을 고르므로, 알맞은 레이블이 없으면 가장 비슷한 다른 레이블로 분류될 수 있다. 예를 들어 `JeekLee는 무신사 플랫폼 엔지니어로 일하고 있다.`에서 `job title` 레이블이 없으면 `무신사 플랫폼 엔지니어`가 `organization`으로 추출된다([기본 사용법](01_basic_usage.ipynb)).

## 텍스트가 엔티티가 되기까지

```text
원문
 │ ① 단어 분할 (word splitter)          "JeekLee는 무신사에서"  →  [JeekLee는] [무신사에서]   (기본: 공백)
 ▼                                                            →  [JeekLee][는] [무신사][에서] (Kiwi 형태소)
단어 토큰 (원문 위치 보존)
 │ ② 서브워드 토큰화 (인코더 토크나이저)
 ▼
서브워드 + 스키마 프롬프트 → ③ 인코더 (DeBERTa / mDeBERTa)
 │
 ▼ ④ 후보 범위 점수 계산: 단어 토큰 경계에서 시작·끝나는 범위만 후보가 된다
 │
 ▼ ⑤ 디코딩: threshold, 중첩 처리, (JointIE) 타입·그래프 제약
 │
결과: {"text", "start", "end", "confidence"}
```

### 단어 분할

GLiNER2는 먼저 문장을 **단어 토큰**으로 나누고, 각 단어를 인코더의 서브워드 토크나이저로 다시 쪼갠다. 엔티티 후보는 단어 토큰의 경계에서만 시작하고 끝난다. 즉 단어 분할이 표현할 수 있는 범위의 상한을 정한다.

기본 분할기 `whitespace`는 공백과 문장부호 기준이며, 공개 체크포인트는 이 분할기로 학습되었다. 한국어는 조사가 명사에 붙으므로 `JeekLee는`이 하나의 단어가 되고, 이 경우 `JeekLee`만 고르는 범위는 후보에 존재하지 않는다. 그래서 모델이 아무리 정확해도 `JeekLee는` 전체를 고를 수밖에 없다.

분할기는 `set_word_splitter()`로 실행 중에 바꿀 수 있다. 내장 분할기는 `whitespace`와 `char`(한 글자 단위, 중국어 등 용)이며, `(token, start, end)`를 반환하는 함수를 직접 넘길 수도 있다. 이 프로젝트는 한국어 형태소 분석기 Kiwi로 분할기를 만들어 사용한다([`config/extractors.py`](../config/extractors.py)). 형태소 분할기를 쓰면 `JeekLee는`이 `JeekLee` + `는`, `무신사에서`가 `무신사` + `에서`로 나뉘어 조사를 뺀 범위가 후보에 들어온다. 반대로 `char`처럼 너무 잘게 나누면 후보 범위가 크게 늘고 학습 때 본 단어 단위와도 달라져, 오히려 결과가 나빠질 수 있다. 세 분할기를 같은 문장들로 비교한 결과는 [단어 분할기 비교](../experiments/02_korean_word_splitter.ipynb)에 있다. 공식 문서도 학습 때와 다른 분할기를 쓰면 품질이 달라질 수 있다고 경고하며, 분할기 선택은 저장되지 않아 체크포인트를 다시 불러오면 `whitespace`로 돌아간다.

### Threshold

| 수준 | 지정 방법 | 의미 |
|---|---|---|
| 호출 전체 | `extract_entities(..., threshold=0.5)` | 이 값 미만의 confidence를 가진 범위를 버린다. 기본값 0.5 |
| 타입·필드별 | 스키마 빌더의 `threshold=` 인자 | 특정 타입이나 필드에만 다른 기준을 적용한다. |
| JointIE | `JointIEConfig(entity_threshold=...)`, 관계별 `threshold=` | 엔티티와 관계 각각의 기준 |

Threshold는 정밀도와 재현율을 맞바꾸는 장치이다. `JeekLee 팀장은 다음 주 부산에서 열리는 AWS 서밋에 카카오 개발자들과 함께 참석한다.`에서 `JeekLee`(person), `부산`(location)처럼 분명한 엔티티는 대개 0.9 이상을 받고, `다음 주`(date)처럼 엔티티로 볼지 애매한 표현은 그보다 낮은 점수를 받는다. 기준을 0.8로 올리면 애매한 표현이 빠져 결과가 깔끔해지지만 필요한 정보도 함께 잃고, 0.2로 내리면 더 많이 찾는 대신 엉뚱한 범위가 섞인다.

다만 threshold는 **점수가 낮은 오류**만 걸러낸다. 모델이 확신하는 오류는 기준을 올려도 남는다. 예를 들어 `JeekLee는 무신사 플랫폼 엔지니어로 일하고 있다.`에서 레이블로 person, organization, location만 주면 `무신사 플랫폼 엔지니어`가 organization으로 0.95를 받는다. 직책을 뜻하는 레이블이 없어서 가장 가까운 organization을 고른 것인데, 점수가 높으니 threshold를 0.9로 올려도 걸러지지 않는다. 이런 오류는 기준을 조정할 게 아니라 레이블 집합, 설명, 타입 제약(JointIE)으로 다뤄야 한다. 두 예시는 [기본 사용법](01_basic_usage.ipynb)에서 직접 실행해 볼 수 있다.

## 관계 추출과 JointIE

IE는 **정보 추출(Information Extraction)**의 약자로, 비정형 텍스트에서 엔티티·관계·속성 같은 구조화된 정보를 뽑아내는 작업 전체를 가리킨다. JointIE(Joint Information Extraction)는 엔티티 추출과 관계 추출을 따로 수행해 이어 붙이지 않고, 한 번에 함께(joint) 결정한다는 뜻이다.

GLiNER2의 관계 추출은 두 가지 방식이 있다.

| 방식 | 호출 | 특징 |
|---|---|---|
| 독립 관계 추출 | `extract_relations()`, 스키마의 `.relations()` | 관계 타입마다 (머리, 꼬리) 범위 쌍을 독립적으로 찾는다. 머리·꼬리의 엔티티 타입은 제약하지 않는다. |
| JointIE | `JointIEEngine.extract(text, schema)` | 엔티티와 관계를 함께 디코딩한다. 관계마다 머리·꼬리 엔티티 타입을 선언하고, 그래프 제약을 만족하는 조합을 빔 탐색으로 찾는다. |

JointIE 스키마는 관계마다 다음 제약을 선언할 수 있다.

| 제약 | 의미 | 예 |
|---|---|---|
| 머리·꼬리 타입 | 허용되는 엔티티 타입 쌍 | `works_for`: person → organization |
| `unique_head` | 한 머리 엔티티는 이 관계를 하나만 가진다. | 현재 직장은 하나 |
| `acyclic` | 관계가 순환을 만들지 않는다. | 자회사 관계 |
| `allow_self`, `no_self_loops` | 자기 자신과의 관계 허용 여부 | |

결과에는 제약을 만족하는 해가 있었는지를 나타내는 `feasible`이 포함된다. 제약이 너무 강해 해가 없으면 빈 결과와 함께 `feasible=False`가 반환되며, 이는 "추출할 사실이 없음"과 다른 의미이다. `neo4j-agent-memory` SDK와 이 프로젝트의 [`GlinerExtractor`](../config/extractors.py)는 POLE+O 온톨로지를 JointIE 스키마로 바꾸어 사용한다.

JointIE도 관계 타입마다 독립적으로 점수를 매기므로, 의미가 겹치는 타입(`LOCATED_AT`과 `RESIDES_AT`)을 하나만 고르지 않는다. 또한 관계는 추출된 엔티티 사이에서만 만들어지므로, 엔티티 범위가 틀리면 관계도 틀린 노드에 연결된다.

## 학습

GLiNER2는 사용자 데이터로 추가 학습할 수 있다. 학습 데이터는 `InputExample` 객체나 JSONL 파일로 준비한다.

```json
{"input": "JeekLee는 무신사에서 일한다.",
 "output": {"entities": {"person": ["JeekLee"], "organization": ["무신사"]},
            "relations": [{"works_for": {"head": "JeekLee", "tail": "무신사"}}]}}
```

같은 문자열이 여러 번 나오거나 위치를 명시해야 하면 `{"text": "JeekLee", "start": 0, "end": 7}`처럼 원문 위치를 줄 수 있다. 분류(`classifications`)와 구조화 레코드(`json_structures`)도 같은 형식으로 학습한다.

| 방식 | 설정 | 학습 대상 | 저장 크기 |
|---|---|---|---|
| 전체 미세조정 | `TrainingConfig(use_lora=False)` | 모든 파라미터 | 체크포인트 전체 |
| LoRA | `TrainingConfig(use_lora=True, lora_r=..., lora_alpha=...)` | 인코더·작업 헤드에 붙인 저차원 어댑터 | 수 MB (multi 기준 약 10MB) |

LoRA 어댑터는 기반 체크포인트와 분리되어 저장되며, `load_adapter()`로 실행 중에 불러오거나 교체할 수 있다. 학습은 `ExtractorTrainer(model, config).train(train_data, eval_data)`로 실행한다.

단어 분할기는 학습 데이터의 수집(collation)에도 적용된다. 따라서 형태소 분할기를 설정한 상태로 학습하면, 모델이 형태소 경계에 맞춰진 범위를 학습한다. 공백 분할기로는 `JeekLee는` 안의 `JeekLee` 같은 정답 범위를 표현할 수 없으므로, 한국어 학습에서는 분할기 선택이 데이터 정합성 문제가 된다. [한국어 미세조정](02_korean_fine_tuning.ipynb)에서 KLUE-NER로 이를 확인한다.

### 한국어 학습 데이터: KLUE-NER

[KLUE](https://klue-benchmark.com/)(Korean Language Understanding Evaluation, 2021)는 한국어 이해 능력을 평가하기 위해 만든 8개 과제의 벤치마크이고, KLUE-NER는 그중 개체명 인식 과제의 데이터셋이다. 라이선스는 CC BY-SA 4.0이며 Hugging Face의 `klue/klue` 저장소에서 받을 수 있다.

| 항목 | 내용 |
|---|---|
| 원문 | 위키트리(WIKITREE) 뉴스 기사 문장(격식체)과 네이버 영화 리뷰(NSMC, 구어체·신조어·오탈자 포함) |
| 규모 | 학습 21,008 / 검증 5,000 / 테스트 5,000문장. 테스트 데이터는 공개되지 않아 검증 데이터로 평가한다. |
| 엔티티 타입 | `PS`(사람), `LC`(장소), `OG`(기관), `DT`(날짜), `TI`(시간), `QT`(수량). 한국정보통신기술협회(TTA) 개체명 지침의 15개 타입 중 MUC-7과 대응되는 6개를 골랐다. |
| 표시 방식 | **문자 단위** BIO 태그 |
| 평가 지표 | 엔티티 단위 macro F1(범위와 타입이 정확히 일치해야 정답), 문자 단위 macro F1(부분 일치 반영) |

문자 단위로 표시한 이유가 이 노트북의 단어 분할 문제와 같다. 원 논문은 한국어의 어절이 `담주가` = `담주` + `가`처럼 내용어와 조사가 결합된 단위이고, 복합어에 공백이 들어가는 경우도 많아 공백 기준 표시가 어렵다고 설명한다. 그래서 KLUE-NER 정답에는 `JeekLee는`이 아니라 `JeekLee`처럼 조사를 뺀 범위가 표시된다. 이 정답으로 GLiNER2를 학습하려면, 정답 범위가 단어 분할기의 토큰 경계와 맞아야 한다.

몇 가지 표기 규칙은 직관과 다르므로 학습·평가 결과를 해석할 때 고려한다.

- `PS`는 개인뿐 아니라 집단의 이름도 포함한다(논문 예시: 밴드 이름 `씨엔블루`).
- 같은 이름도 문맥에 따라 타입이 달라진다. 잡지 이름이 "서점에서 사서 읽었다"는 문맥에 나오면 기관(`OG`)으로 표시하지 않는다.
- 사람 이름은 공개된 인물 위주로 남아 있다. 개인정보 보호를 위해 검색 엔진에 나오지 않는 인물 이름이 들어간 문장은 제거했다.

## LLM 추출과의 관계

이 프로젝트의 실험 결과를 요약하면 다음과 같다.

| 관점 | GLiNER2.5 multi + 형태소 분할기 | LLM (`gpt-5.4-mini`) |
|---|---|---|
| 엔티티 정확 일치 (참고 정답 21개) | 16 | 19.7 |
| 관계 | 적중 수는 비슷하나, 문장에 없는 관계와 중복 타입이 섞인다. | 문맥을 해석하지만 회차마다 결과가 달라진다. |
| 지연 시간 | 문장당 약 0.14초 (CPU) | 문장당 약 1.8초 (네트워크 포함) |
| 결정성 | 같은 입력에 항상 같은 결과 | 회차마다 변동 |
| 비용·데이터 반출 | 없음 (로컬 실행) | API 비용, 외부 전송 |

GLiNER2의 강점은 품질 상한보다는 운영 특성에 있다. 처리량이 크거나, 결과가 결정적이어야 하거나, 데이터를 외부로 보낼 수 없는 환경에서 현실적인 선택지이다. 품질 격차는 대상 언어와 도메인의 데이터로 학습하여 줄일 수 있으며, 그 방법을 [한국어 미세조정](02_korean_fine_tuning.ipynb)에서 다룬다.

## 활용 아이디어

아직 실험하지 않았지만, 지금까지의 실험에서 나온 활용 방향을 정리한다.

### 파인튜닝은 온톨로지의 구현이다

GLiNER의 레이블을 온톨로지 엔티티 타입의 출발점으로 보면, 파인튜닝은 **환경에 맞춘 온톨로지를 추출기에 구현하는 일**로 해석할 수 있다. 온톨로지의 타입에는 두 층이 있다.

| 층 | 예 | GLiNER에서 |
|---|---|---|
| 이름과 정의 | `person`: "개인 또는 집단의 고유한 이름" | 레이블 이름과 설명 |
| 실제 경계 (무엇이 그 타입의 사례인가) | `박씨`는 `박`인가 `박씨`인가, `북한`은 장소인가 기관인가 | 학습된 가중치 |

같은 레이블 이름이라도 무엇으로 학습했는지에 따라 경계가 달라진다. 사람이 만든 KLUE 정답으로 학습한 어댑터와 LLM이 만든 정답으로 학습한 어댑터는 둘 다 `person`/`organization`/`location`을 쓰지만, 호칭 처리, 주소 범위, 국가 타입을 서로 다르게 판단한다([LLM 증류](../experiments/03_llm_distillation.ipynb)).

다만 어댑터를 온톨로지 자체로 볼 수는 없다. 온톨로지는 명시적이고(읽고 검토하고 버전 관리할 수 있다), 상하위 타입·관계 제약·추론을 표현하며, 저장·검색·병합 규칙의 기준이 된다. 어댑터는 그 경계를 가중치 안에 암묵적으로 담을 뿐이고, 학습 데이터의 문체와 도메인까지 함께 배운다. 따라서 코드와 빌드 결과물의 관계로 다루는 것이 안전하다.

```text
온톨로지 (타입·정의·관계·제약)        ← 원본, 기준
   ↓ 경계 사례를 구체화
표기 지침 (호칭, 주소 범위, 국가 타입…)
   ↓ 사람 정답 또는 LLM 선생님 프롬프트
학습 데이터
   ↓ LoRA 파인튜닝
어댑터                               ← 해당 온톨로지 버전의 빌드 결과물
   ↓ 같은 지침으로 만든 평가 데이터로 검증
```

온톨로지 버전마다 어댑터를 다시 만들고, 어댑터가 온톨로지를 제대로 구현했는지는 같은 지침으로 만든 평가 데이터로 확인한다.

### LLM 없이 엔티티와 관계를 한 번에 추출

JointIE는 온톨로지의 엔티티 타입, 관계 타입, 관계의 출발·도착 타입, 관계 제약(`unique_head`, `acyclic`, `no_self_loops`)을 스키마로 받아 엔티티와 관계를 한 번의 추출로 낸다. 기능만 보면 LLM 없이도 온톨로지 기반 지식 그래프 추출이 가능하다. 남은 문제는 품질이며, 특히 관계가 약하다.

| 한계 | 내용 |
|---|---|
| 관계 품질 | 공개 체크포인트의 zero-shot 관계는 문장에 없는 관계를 만들기도 하고, 틀린 관계의 confidence가 맞은 관계보다 높을 수 있어 threshold로 거르기 어렵다. |
| 관계 미학습 | 지금까지의 미세조정은 엔티티만 다뤘다. KLUE-NER에는 관계 정답이 없다. |
| 문장 밖 정보 | 지시어(`그는`, `그 회사`)를 앞 문장 엔티티와 연결하거나, 명시되지 않은 사실을 추론하지 못한다. |

### 관계 증류

`gliner2`의 학습 데이터는 엔티티와 함께 관계도 받는다(`InputExample(..., relations=[Relation("works_for", head="JeekLee", tail="무신사")])`). 따라서 [LLM 증류](../experiments/03_llm_distillation.ipynb)의 파이프라인을 관계로 확장할 수 있다.

1. 선생님 LLM에 온톨로지의 관계 타입(이름, 정의, 출발·도착 타입)을 주고 엔티티와 관계를 함께 표시하게 한다.
2. 끝점이 엔티티 목록에 없거나 타입 규칙에 어긋나는 관계는 버리고, 나머지를 `Relation`으로 변환한다.
3. 엔티티와 관계를 함께 LoRA로 학습하고, 관계 정답이 있는 평가 데이터로 학습 전·후와 LLM을 비교한다.

실험 전에 확인할 점이 있다.

- 학습 형식의 관계는 GLiNER2의 일반 관계 추출 작업 기준이다. 학습 효과가 JointIE 추론에도 반영되는지 확인해야 한다.
- 관계의 머리·꼬리는 문자열로만 지정하므로, 같은 이름이 한 문장에 여러 번 나오면 모호하다.
- "이 문장에는 이 관계가 없다"는 부정 예시를 어떻게 가르치는지 확인해야 한다. 문장에 없는 관계를 만드는 문제와 직결된다.
- 관계는 엔티티보다 정답 기준이 더 모호하다(명시된 사실만인지, 추론까지인지). 온톨로지 기준으로 관계 정답을 단 평가 문장이 먼저 필요하다.

### agent-memory 파이프라인에 적용

`neo4j-agent-memory` SDK의 추출 파이프라인은 spaCy → GLiNER → LLM 단계를 엮고 결과를 합친다. 확인한 동작은 다음과 같다(SDK 0.8.0).

| 항목 | 동작 |
|---|---|
| 관계를 추출하는 단계 | GLiNER(JointIE)와 LLM. spaCy는 엔티티만 낸다. |
| 엔티티 합치기 | `CONFIDENCE` 방식이면 같은 이름·타입 중 confidence가 높은 쪽을 남긴다. spaCy는 confidence를 고정값 0.85로 붙인다. |
| 관계 합치기 | 단계별 결과를 모아 (출발, 관계, 도착) 단위로 중복만 제거한다(사실상 합집합). 이후 온톨로지 타입 검증을 통과하지 못한 관계를 버린다. |
| "LLM fallback" | `fallback_on_empty=True`이면 앞 단계 결과와 관계없이 **모든 단계가 매번 실행**된다(`stop_on_success=False`). |

LLM 비용을 줄이면서 한국어 품질을 확보하는 구성은 다음과 같다.

```text
GLiNER (multi + Kiwi 분할기 + 온톨로지 기준으로 증류한 어댑터) ← 대부분의 문장
   └─ 결과가 비었거나 confidence가 낮은 문장만 LLM              ← 비용을 통제하는 fallback
```

SDK의 GLiNER 단계는 체크포인트 이름만 받으므로, 분할기와 어댑터를 쓰려면 직접 만든 추출기를 파이프라인에 연결해야 할 수 있다. 이 부분은 아직 확인하지 않았다.

## 참고 자료

- [GLiNER2 저장소와 README](https://github.com/fastino-ai/GLiNER2)
- [GLiNER: Generalist Model for Named Entity Recognition using Bidirectional Transformer](https://arxiv.org/abs/2311.08526)
- [Introducing GLiNER2.5](https://fastino.ai/blog/gliner2-5-span-free-information-extraction)
- [GLiNER2 family (Hugging Face)](https://huggingface.co/collections/fastino/gliner2-family)
- [Kiwi 한국어 형태소 분석기](https://github.com/bab2min/kiwipiepy)
- [KLUE: Korean Language Understanding Evaluation](https://arxiv.org/abs/2105.09680) (3.4절 KLUE-NER), [klue/klue 데이터셋](https://huggingface.co/datasets/klue/klue)